<a href="https://colab.research.google.com/github/prithwis/fintech/blob/main/MNA_PureMomentum_Upload_NSE12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PureMomentum — Monday Night Analysis (MNA)

**Input:** Upload the `NSE12_YYYYMMDD.zip` generated by `Get-NSE12.ps1` (a ZIP containing 12 daily NSE CM-UDiFF ZIP files). **No NSE network calls.**

**Screen:** latest 10 completed sessions; ordinary `EQ` securities; positive OLS slope of log closing price; **R² > 0.80**; average daily actual traded value ≥ **₹1 crore**; rank by descending slope. Requires all 10 observations for each ISIN. Prices are *unadjusted*; conservative anomaly checks exclude suspected corporate actions. Tuesday execution prices may differ from Monday's closing prices.


In [1]:
import io, re, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from google.colab import files

WINDOW = 10
MIN_R2 = 0.80
MIN_AVG_TRADED_VALUE = 1_00_00_000  # rupees per session
WEEKLY_BUDGET = 10_000             # edit as needed
MAX_POSITION_PCT = 0.50           # approximately equal allocation to two names
MAX_ABS_DAILY_MOVE = 0.40         # flag extreme close-to-close jumps (possible corporate actions)

print('Upload NSE12_YYYYMMDD.zip (the outer ZIP):')
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one NSE12 ZIP file.')
outer_name, outer_bytes = next(iter(uploaded.items()))
print(f'Uploaded: {outer_name} ({len(outer_bytes):,} bytes)')

Upload NSE12_YYYYMMDD.zip (the outer ZIP):


Saving NSE12_20261106.zip to NSE12_20261106.zip
Uploaded: NSE12_20261106.zip (2,484,707 bytes)


In [2]:
# Validate all nested ZIPs and parse their NSE CM-UDiFF CSVs.
filename_re = re.compile(r'^BhavCopy_NSE_CM_0_0_0_(\d{8})_F_0000\.csv\.zip$', re.I)
required = {'TradDt','ISIN','TckrSymb','SctySrs','ClsPric','PrvsClsgPric','TtlTrfVal','TtlTradgVol'}
frames, audit = [], []
with zipfile.ZipFile(io.BytesIO(outer_bytes)) as outer:
    nested = [i for i in outer.infolist() if not i.is_dir()]
    if len(nested) != 12:
        raise ValueError(f'Expected 12 daily ZIPs, found {len(nested)} files in outer archive.')
    for member in nested:
        name = Path(member.filename).name
        m = filename_re.fullmatch(name)
        if not m:
            raise ValueError(f'Unexpected file in outer archive: {member.filename}')
        expected = pd.to_datetime(m.group(1), format='%Y%m%d')
        with zipfile.ZipFile(io.BytesIO(outer.read(member))) as inner:
            csvs = [i for i in inner.infolist() if not i.is_dir() and i.filename.lower().endswith('.csv')]
            if len(csvs) != 1:
                raise ValueError(f'{name}: expected exactly one CSV, found {len(csvs)}')
            with inner.open(csvs[0]) as f:
                df = pd.read_csv(f, low_memory=False)
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f'{name}: missing columns {sorted(missing)}')
        df['date'] = pd.to_datetime(df['TradDt'], errors='coerce').dt.normalize()
        if df.empty or df['date'].isna().any() or df['date'].nunique() != 1 or df['date'].iloc[0] != expected:
            raise ValueError(f'{name}: CSV date mismatch or invalid dates')
        if expected > pd.Timestamp.now(tz='Asia/Kolkata').tz_localize(None).normalize():
            raise ValueError(f'{name}: future session date')
        audit.append({'date': expected, 'filename': name, 'all_rows': len(df),
                      'eq_rows': int(df['SctySrs'].eq('EQ').sum())})
        frames.append(df)

session_audit = pd.DataFrame(audit).sort_values('date').reset_index(drop=True)
if session_audit['date'].duplicated().any():
    raise ValueError('Duplicate session dates in archive')
if len(session_audit) != 12:
    raise ValueError('Expected 12 distinct trading sessions')
if (session_audit['eq_rows'] < 100).any():
    raise ValueError('Unexpectedly few EQ rows in a session; inspect archive')
print('12 distinct sessions validated; latest:', session_audit.date.max().date())
display(session_audit)
raw = pd.concat(frames, ignore_index=True)


12 distinct sessions validated; latest: 2026-11-06


,date,filename,all_rows,eq_rows
0,2026-10-22,BhavCopy_NSE_CM_0_0_0_20261022_F_0000.csv.zip,3649,2694
1,2026-10-23,BhavCopy_NSE_CM_0_0_0_20261023_F_0000.csv.zip,3730,2706
2,2026-10-26,BhavCopy_NSE_CM_0_0_0_20261026_F_0000.csv.zip,3686,2709
3,2026-10-27,BhavCopy_NSE_CM_0_0_0_20261027_F_0000.csv.zip,3670,2712
4,2026-10-28,BhavCopy_NSE_CM_0_0_0_20261028_F_0000.csv.zip,3682,2711
5,2026-10-29,BhavCopy_NSE_CM_0_0_0_20261029_F_0000.csv.zip,3697,2712
6,2026-10-30,BhavCopy_NSE_CM_0_0_0_20261030_F_0000.csv.zip,3641,2703
7,2026-11-02,BhavCopy_NSE_CM_0_0_0_20261102_F_0000.csv.zip,3715,2714
8,2026-11-03,BhavCopy_NSE_CM_0_0_0_20261103_F_0000.csv.zip,3717,2744
9,2026-11-04,BhavCopy_NSE_CM_0_0_0_20261104_F_0000.csv.zip,3705,2739


In [3]:
# Select last 10 sessions and construct a clean EQ-only price panel.
sessions = list(session_audit['date'].tail(WINDOW))
eq = raw.loc[raw['date'].isin(sessions) & raw['SctySrs'].eq('EQ')].copy()
eq['isin'] = eq['ISIN'].astype('string').str.strip()
eq['symbol'] = eq['TckrSymb'].astype('string').str.strip()
for src, dst in [('ClsPric','close'),('PrvsClsgPric','prev_close'),
                 ('TtlTrfVal','traded_value'),('TtlTradgVol','volume')]:
    eq[dst] = pd.to_numeric(eq[src], errors='coerce')
if eq.duplicated(['date','isin']).any():
    dup = eq.loc[eq.duplicated(['date','isin'], keep=False), ['date','isin','symbol']].head()
    display(dup)
    raise ValueError('Duplicate EQ ISIN/date observations; investigate before screening.')
# Basic record validation; flag suspected unadjusted price discontinuities.
eq['bad'] = (eq['isin'].isna() | (eq['isin'].str.len() != 12) |
             eq['symbol'].isna() | ~np.isfinite(eq['close']) | (eq['close'] <= 0) |
             ~np.isfinite(eq['prev_close']) | (eq['prev_close'] <= 0) |
             ~np.isfinite(eq['traded_value']) | (eq['traded_value'] < 0) |
             ~np.isfinite(eq['volume']) | (eq['volume'] <= 0))
eq['daily_move'] = eq['close'] / eq['prev_close'] - 1
eq['suspect_jump'] = eq['daily_move'].abs() > MAX_ABS_DAILY_MOVE
# Check previous-close continuity against prior session close for each ISIN;
# small deviations may reflect adjustments or corrections, so only flag > 10%.
eq = eq.sort_values(['isin','date'])
eq['prior_observed_close'] = eq.groupby('isin')['close'].shift()
eq['prior_observed_date'] = eq.groupby('isin')['date'].shift()
eq['sid'] = eq['date'].map({d:i for i,d in enumerate(sessions)})
eq['prior_sid'] = eq.groupby('isin')['sid'].shift()
consecutive = eq['sid'].sub(eq['prior_sid']).eq(1)
eq['suspect_discontinuity'] = (consecutive &
    ((eq['prev_close'] / eq['prior_observed_close'] - 1).abs() > 0.10))
eq['bad'] = eq['bad'] | eq['suspect_jump'] | eq['suspect_discontinuity']
print('EQ rows:',len(eq),'| flagged rows:',int(eq['bad'].sum()))
print('Screen dates:',sessions[0].date(),'to',sessions[-1].date())

EQ rows: 27232 | flagged rows: 4
Screen dates: 2026-10-26 to 2026-11-06


In [4]:
# OLS on log(close) for every ISIN with exactly 10 clean consecutive observations.
x = np.arange(WINDOW, dtype=float)
xc = x - x.mean()
sxx = np.dot(xc, xc)
records = []
exclusions = {'incomplete_or_flagged': 0, 'nonpositive_slope': 0, 'low_r2': 0, 'low_liquidity': 0}
for isin, g in eq.groupby('isin', sort=False):
    g = g.sort_values('date')
    if len(g) != WINDOW or g['sid'].tolist() != list(range(WINDOW)) or g['bad'].any():
        exclusions['incomplete_or_flagged'] += 1
        continue
    y = np.log(g['close'].to_numpy(dtype=float))
    yc = y - y.mean()
    slope = float(np.dot(xc, yc) / sxx)
    if slope <= 0:
        exclusions['nonpositive_slope'] += 1
        continue
    resid = yc - slope * xc
    sst = float(np.dot(yc, yc))
    r2 = float(1 - np.dot(resid, resid) / sst) if sst > 0 else 0.0
    if not r2 > MIN_R2:
        exclusions['low_r2'] += 1
        continue
    avg_value = float(g['traded_value'].mean())
    if avg_value < MIN_AVG_TRADED_VALUE:
        exclusions['low_liquidity'] += 1
        continue
    last = g.iloc[-1]
    records.append({'symbol':last['symbol'], 'isin':isin, 'close':float(last['close']),
                    'slope':slope, 'r2':r2, 'avg_traded_value':avg_value,
                    'avg_volume':float(g['volume'].mean()),
                    'ten_session_return_pct':100*(float(last['close'])/float(g.iloc[0]['close'])-1),
                    'approx_daily_growth_pct':100*np.expm1(slope)})
ranked = pd.DataFrame.from_records(records, columns=[
    'symbol','isin','close','slope','r2','avg_traded_value','avg_volume',
    'ten_session_return_pct','approx_daily_growth_pct'])
ranked = ranked.sort_values(['slope','r2'], ascending=[False,False]).reset_index(drop=True)
ranked.index = ranked.index + 1
ranked.index.name = 'rank'
print('Qualified:',len(ranked),'| Exclusions:',exclusions)
display(ranked.head(30).round({'close':2,'slope':5,'r2':3,'avg_traded_value':0,
                                'avg_volume':0,'ten_session_return_pct':2,'approx_daily_growth_pct':2}))

Qualified: 295 | Exclusions: {'incomplete_or_flagged': 123, 'nonpositive_slope': 597, 'low_r2': 1573, 'low_liquidity': 189}


,symbol,isin,close,slope,r2,avg_traded_value,avg_volume,ten_session_return_pct,approx_daily_growth_pct
rank,,,,,,,,,
1,HEGAM,INE545A01024,293.87,0.04008,0.896,1.473794e+09,5412462.0,43.35,4.09
2,RAMKY,INE874I01013,392.00,0.03991,0.909,4.633207e+08,1433238.0,48.88,4.07
3,SUPREMEINF,INE550H01011,70.71,0.03896,0.848,1.315614e+07,211835.0,31.09,3.97
4,BOROLTD,INE02PY01013,318.20,0.03875,0.816,8.151966e+08,2838278.0,34.06,3.95
5,FAZE3Q,INE963C01033,550.00,0.03854,0.809,1.595000e+08,304744.0,33.24,3.93
6,NELCAST,INE189I01024,157.32,0.03679,0.860,2.707204e+08,1705690.0,29.96,3.75
7,AGIIL,INE976R01033,300.93,0.03547,0.902,1.139091e+09,4458944.0,50.05,3.61
8,RUSHIL,INE573K01025,19.30,0.03297,0.837,2.250655e+07,1260783.0,30.94,3.35
9,VEDPOWER,INE694L01019,35.33,0.03189,0.864,2.552906e+09,75362719.0,25.24,3.24


In [9]:
# PureMomentum — Broad Market Benchmark

valid = eq.loc[~eq['bad']].sort_values(['isin', 'date'])

returns = (
    valid.groupby('isin')
    .agg(
        sessions=('close', 'count'),
        first_close=('close', 'first'),
        last_close=('close', 'last')
    )
)

returns = returns[returns['sessions'] == WINDOW].copy()

returns['return_pct'] = (
    (returns['last_close'] / returns['first_close'] - 1) * 100
)

r = returns['return_pct']

print(f"Stocks analysed       : {len(r)}")
print(f"Median 10-session return: {r.median():.2f}%")
print(f"Mean 10-session return  : {r.mean():.2f}%")
print(f"Stocks with gains       : {(r > 0).mean()*100:.1f}%")
print(f"Stocks with gains >10%  : {(r > 10).mean()*100:.1f}%")

Stocks analysed       : 2654
Median 10-session return: 5.00%
Mean 10-session return  : 5.05%
Stocks with gains       : 78.5%
Stocks with gains >10%  : 19.4%


In [7]:
# PureMomentum — Top 10 candidates

display(ranked.head(10).round({
    'close': 2,
    'slope': 5,
    'r2': 3,
    'avg_traded_value': 0,
    'avg_volume': 0,
    'ten_session_return_pct': 2,
    'approx_daily_growth_pct': 2
}))

,symbol,isin,close,slope,r2,avg_traded_value,avg_volume,ten_session_return_pct,approx_daily_growth_pct
rank,,,,,,,,,
1,HEGAM,INE545A01024,293.87,0.04008,0.896,1.473794e+09,5412462.0,43.35,4.09
2,RAMKY,INE874I01013,392.00,0.03991,0.909,4.633207e+08,1433238.0,48.88,4.07
3,SUPREMEINF,INE550H01011,70.71,0.03896,0.848,1.315614e+07,211835.0,31.09,3.97
4,BOROLTD,INE02PY01013,318.20,0.03875,0.816,8.151966e+08,2838278.0,34.06,3.95
5,FAZE3Q,INE963C01033,550.00,0.03854,0.809,1.595000e+08,304744.0,33.24,3.93
6,NELCAST,INE189I01024,157.32,0.03679,0.860,2.707204e+08,1705690.0,29.96,3.75
7,AGIIL,INE976R01033,300.93,0.03547,0.902,1.139091e+09,4458944.0,50.05,3.61
8,RUSHIL,INE573K01025,19.30,0.03297,0.837,2.250655e+07,1260783.0,30.94,3.35
9,VEDPOWER,INE694L01019,35.33,0.03189,0.864,2.552906e+09,75362719.0,25.24,3.24


In [10]:
# Indicative Tuesday order plan (NOT executable orders).
# Allocate at most half of the weekly budget to each of two affordable names.
# No redistribution of unused cash; actual Tuesday price may differ.
allocation = WEEKLY_BUDGET * MAX_POSITION_PCT
picks = []
for rank, row in ranked.iterrows():
    qty = int(allocation // row['close'])
    if qty < 1:
        continue
    picks.append({'rank':rank, 'symbol':row['symbol'], 'isin':row['isin'],
                  'reference_close':row['close'], 'indicative_qty':qty,
                  'estimated_cost':qty*row['close'], 'slope':row['slope'], 'r2':row['r2']})
    if len(picks) == 2:
        break
orders = pd.DataFrame(picks)
spent = float(orders['estimated_cost'].sum()) if len(orders) else 0.0
print(f'Weekly allocation: ₹{WEEKLY_BUDGET:,.2f} | Estimated spend: ₹{spent:,.2f} | Residual: ₹{WEEKLY_BUDGET-spent:,.2f}')
if len(orders) < 2:
    print('WARNING: Fewer than two affordable qualifying candidates.')
display(orders.round(4))
print('IMPORTANT: use actual Tuesday execution prices to record positions; Monday close is only a sizing reference.')

Weekly allocation: ₹10,000.00 | Estimated spend: ₹9,699.79 | Residual: ₹300.21


,rank,symbol,isin,reference_close,indicative_qty,estimated_cost,slope,r2
0,1,HEGAM,INE545A01024,293.87,17,4995.79,0.0401,0.8960
1,2,RAMKY,INE874I01013,392.00,12,4704.00,0.0399,0.9088


IMPORTANT: use actual Tuesday execution prices to record positions; Monday close is only a sizing reference.


In [12]:
# Save outputs and download a single results bundle.
from datetime import datetime
stamp = sessions[-1].strftime('%Y%m%d')
paths = {
    f'PureMomentum_ranked_{stamp}.csv': ranked.reset_index().to_csv(index=False),
    f'PureMomentum_orders_{stamp}.csv': orders.to_csv(index=False),
    f'PureMomentum_session_audit_{stamp}.csv': session_audit.to_csv(index=False),
}
notes = (f'Latest session: {sessions[-1].date()}\nFormation window: {WINDOW} sessions\n'
         f'Slope > 0; R2 > {MIN_R2}; mean TtlTrfVal >= INR {MIN_AVG_TRADED_VALUE:,}\n'
         f'Candidates: {len(ranked)}; exclusion counts: {exclusions}\n'
         'Unadjusted closes; suspected discontinuities flagged, not corporate-action-adjusted.\n'
         'Indicative sizing at last close; excludes charges, slippage and Tuesday price changes.\n'
         'Weekly exit/holding rules require separately recorded actual fills and subsequent prices.\n')
paths[f'PureMomentum_notes_{stamp}.txt'] = notes
#out = f'PureMomentum_results_{stamp}.zip'
out = f'NSE12_{stamp}_Output.zip'



with zipfile.ZipFile(out, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for filename, contents in paths.items():
        z.writestr(filename, contents)
print('Saved:',out)
files.download(out)

Saved: NSE12_20261106_Output.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Interpretation and limitations

- This is a **Monday-close screen**, not a guarantee of Tuesday availability or execution price.
- The latest **10 of the 12** supplied sessions are used. The extra two sessions provide acquisition redundancy, not additional regression observations.
- `TtlTrfVal` is the **actual daily traded value in rupees**, averaged over the ten sessions.
- A 40% daily move or a 10% previous-close discontinuity is conservatively excluded as suspicious, **not proven** to be a corporate action. Other splits, bonuses or adjustments may escape these checks. Inspect unusual names before trading.
- The notebook does **not** automatically enforce the one-week/second-week exit rules: those require a persistent position ledger with actual Tuesday fills and later prices.
- No transaction charges, taxes or slippage are included in the indicative allocation.